# GenLab — Launcher
Plataforma modular para ejecutar modelos open source de IA en Colab.

**Antes de ejecutar:**
1. Concede permisos de Drive cuando se solicite.
2. Configura `HF_TOKEN` en Secrets de Colab (Icono llave → `HF_TOKEN`).
3. Asegúrate de usar runtime **T4 GPU** (Entorno de ejecución → Cambiar tipo de entorno de ejecución).

In [ ]:
# 1. Instalar dependencias
!pip install -q torch diffusers transformers huggingface-hub imageio imageio-ffmpeg accelerate psutil pyyaml

In [ ]:
# 2. Montar Google Drive (para outputs persistentes)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clonar repo en /content (no Drive — evita problemas FUSE)
REPO_URL = 'https://github.com/ke1npro/GenLab.git'

import os, sys
GENLAB_SRC = '/content/genlab'

if os.path.isdir(GENLAB_SRC):
    !git -C $GENLAB_SRC pull
else:
    !git clone $REPO_URL $GENLAB_SRC

sys.path.insert(0, f'{GENLAB_SRC}/src')
os.chdir(GENLAB_SRC)

import genlab
print(f'genlab OK: {genlab.__file__}')

In [ ]:
# 4. Bootstrap — diagnóstico del entorno
from genlab import GenLab
info = GenLab.bootstrap()

In [ ]:
# 5. Configurar token de Hugging Face
from google.colab import userdata
import os
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN cargado desde Secrets de Colab.')
except Exception:
    token = input('Ingresa tu token de Hugging Face (Enter para omitir): ').strip()
    if token:
        os.environ['HF_TOKEN'] = token
    else:
        print('HF_TOKEN no configurado.')

In [ ]:
# 6. Generar video
result = GenLab().run(
    task='text_to_video',
    model='cogvideo',
    prompt='Un astronauta montando un caballo en la luna, estilo cinematográfico',
    config={'model': {'steps': 50, 'fps': 8, 'frames': 49}}
)
print(f'Video generado: {result["video_path"]}')

In [ ]:
# 7. Mostrar resultado
from IPython.display import Video
Video(result['video_path'], width=480)